# Лабораторная работа № 1.1 — LU-разложение с выбором главного элемента

Вариант 6. 

Нужно решить Ax=b, найти определитель и обратную матрицу. Выбор главного элемента меняет порядок строк: PA=LU.

## Исходные данные

Числа ниже соответствуют файлу input.txt этой работы.

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from typing import List

rows = [[1.0, 2.0, -1.0, -7.0], [8.0, 0.0, -9.0, -3.0], [2.0, -3.0, 7.0, 1.0], [1.0, -5.0, -6.0, 8.0], [-23.0, 39.0, -7.0, 30.0]]
A, b = rows[:-1], rows[-1]
print('A =', np.array(A), '\nb =', b)

A = [[ 1.  2. -1. -7.]
 [ 8.  0. -9. -3.]
 [ 2. -3.  7.  1.]
 [ 1. -5. -6.  8.]] 
b = [-23.0, 39.0, -7.0, 30.0]


## Перестановки строк

На каждом шаге выбираем максимальный по модулю элемент текущего столбца. Вспомогательное исключение определяет итоговую матрицу перестановки P.

In [2]:
def permutation_matrix(A):
    """Выбор главного элемента на текущем шаге исключения."""
    n = len(A)
    P = [[int(i == j) for j in range(n)] for i in range(n)]
    U = [row[:] for row in A]
    swaps = 0
    for i in range(n):
        max_row = max(range(i, n), key=lambda k: abs(U[k][i]))
        if U[max_row][i] == 0:
            raise ValueError("Singular matrix: a zero pivot was encountered")
        if max_row != i:
            U[i], U[max_row] = U[max_row], U[i]
            P[i], P[max_row] = P[max_row], P[i]
            swaps += 1
        for j in range(i + 1, n):
            multiplier = U[j][i] / U[i][i]
            for k in range(i, n):
                U[j][k] -= multiplier * U[i][k]
    return P, swaps


def matrix_mult(A, B):
    n = len(A)
    return [[sum(A[i][k] * B[k][j] for k in range(n)) for j in range(n)] for i in range(n)]

In [3]:
P, swaps = permutation_matrix(A)
PA = matrix_mult(P, A)
print('P =\n', np.array(P), '\nПерестановок:', swaps)

P =
 [[0 1 0 0]
 [0 0 0 1]
 [0 0 1 0]
 [1 0 0 0]] 
Перестановок: 2


## Составляем L и U

Множитель U[j][i]/U[i][i] записывается в L. Вычитание соответствующей доли ведущей строки обнуляет элемент в U.

In [4]:
def LU_decompose(PA):
    n = len(PA)
    # lower
    L = [[0 for _ in range(n)] for _ in range(n)]
    # upper
    U = [row[:] for row in PA]

    for i in range(n):
        # Заполняем матрицы L и U
        L[i][i] = 1
        if U[i][i] == 0:
            raise ValueError("Singular matrix: a zero pivot was encountered")
        for j in range(i + 1, n):
            if U[i][i] != 0:
                L[j][i] = U[j][i] / U[i][i]
                for k in range(i, n):
                    U[j][k] -= L[j][i] * U[i][k]

    return L, U

In [5]:
L, U = LU_decompose(PA)
print('L =\n', np.array(L), '\nU =\n', np.array(U))

L =
 [[ 1.          0.          0.          0.        ]
 [ 0.125       1.          0.          0.        ]
 [ 0.25        0.6         1.          0.        ]
 [ 0.125      -0.4        -0.14989733  1.        ]] 
U =
 [[ 8.          0.         -9.         -3.        ]
 [ 0.         -5.         -4.875       8.375     ]
 [ 0.          0.         12.175      -3.275     ]
 [ 0.          0.          0.         -3.76591376]]


## Решение, определитель и обратная матрица

Сначала решаем Ly=Pb, затем Ux=y. Определитель равен произведению диагональных элементов U с учётом перестановок. Для обратной матрицы решаем систему с каждым единичным столбцом.

In [6]:
def solve(L, U, b):
    n = len(L)
    # L * y = b
    y = [0 for _ in range(n)]
    for i in range(n):
        y[i] = (b[i] - sum(L[i][j] * y[j] for j in range(i))) / L[i][i]

    # U * x = y
    x = [0 for _ in range(n)]
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - sum(U[i][j] * x[j] for j in range(i + 1, n))) / U[i][i]
    return x


def transpose(A):
    m = len(A)
    n = len(A[0])
    A_T = [[A[j][i] for j in range(n)] for i in range(m)]
    return A_T


def inverse_matrix(A):
    n = len(A)
    E = [[1 if (i == j) else 0 for j in range(n)] for i in range(n)]

    P, _ = permutation_matrix(A)
    PA = matrix_mult(P, A)
    L, U = LU_decompose(PA)

    A_inv = []
    for i in range(n):
        Pb = [sum(P[j][k] * E[k][i] for k in range(n)) for j in range(n)]
        row_inv = solve(L, U, Pb)
        A_inv.append(row_inv)
    return transpose(A_inv)


def determinant(L, U):
    n = len(U)
    det = 1
    for i in range(n):
        det *= U[i][i]
    return det


def matrix_vector_mult(A, x):
    n = len(A)
    m = len(x)
    return [sum(A[i][k] * x[k] for k in range(m)) for i in range(n)]

In [7]:
Pb = matrix_vector_mult(P, b)
x = solve(L, U, Pb)
det = (-1)**swaps * determinant(L, U)
A_inv = inverse_matrix(A)
print('x =', np.round(x, 8), '\ndet(A) =', round(det, 8))

x = [ 6.  6. -1.  6.] 
det(A) = 1834.0


## Самопроверка

Перемножь L и U. Получится ли PA? Проверь также Ax=b и AA⁻¹=I.

In [8]:
assert np.allclose(np.array(L) @ np.array(U), PA)
assert np.allclose(np.array(A) @ x, b)
assert np.allclose(np.array(A) @ np.array(A_inv), np.eye(len(A)))
print('max|PA-LU| =', np.max(np.abs(np.array(PA)-np.array(L)@np.array(U))))
print('max|Ax-b| =', np.max(np.abs(np.array(A)@x-b)))

max|PA-LU| = 8.881784197001252e-16
max|Ax-b| = 7.105427357601002e-15
